# 02 Article Contour Follow Panel

This notebook launches the new contour-based driving panel from `scripts/teleop_server_article_contour_panel.py`.

It reuses:

- `.jetcar_motor_calibration.json` from notebook `01`
- `.contour_follow_settings.json` for the detection sliders

Use this after motor calibration is already saved.


## Safety

Start on a stand, then move to the track only after camera, serial, and test preset buttons all look correct. Keep `Stop Auto` or `Stop Motors` ready.


In [8]:
import glob
import os
import socket
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
video_devices = sorted(glob.glob('/dev/video*'))
serial_candidates = [p for p in ('/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]
hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial candidates:', serial_candidates or ['none found'])
print('Calibration file exists:', (project_root / '.jetcar_motor_calibration.json').exists())
print('Contour settings file exists:', (project_root / '.contour_follow_settings.json').exists())
print('Host IPv4:', ip_addresses or ['127.0.0.1'])


Project root: /home/orin/JetCar
Video devices: ['/dev/video0']
Serial candidates: ['/dev/ttyTHS1', '/dev/ttyTHS2']
Calibration file exists: True
Contour settings file exists: False
Host IPv4: ['127.0.1.1']


In [9]:
camera_source = 'csi'
host = '0.0.0.0'
http_port = 8765
serial_port = serial_candidates[0] if serial_candidates else '/dev/ttyTHS1'
baudrate = 115200
sensor_id = 0
device_index = 0
frame_width = 1280
frame_height = 720
warmup_frames = 12

print('Edit the camera or port values here if needed, then run the next cells.')


Edit the camera or port values here if needed, then run the next cells.


In [10]:
import shlex
import subprocess
import sys
import time
from IPython.display import HTML, display

if '_article_contour_process' not in globals():
    _article_contour_process = None

panel_log_path = project_root / '.teleop_server_article_contour_panel.notebook.log'

def panel_python() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable


def panel_command() -> list[str]:
    return [
        panel_python(),
        str(project_root / 'scripts' / 'teleop_server_article_contour_panel.py'),
        '--host', host,
        '--http-port', str(http_port),
        '--camera-source', camera_source,
        '--sensor-id', str(sensor_id),
        '--device-index', str(device_index),
        '--width', str(frame_width),
        '--height', str(frame_height),
        '--warmup-frames', str(warmup_frames),
        '--port', serial_port,
        '--baudrate', str(baudrate),
    ]


def stop_article_contour_panel() -> None:
    global _article_contour_process
    if _article_contour_process is None:
        return
    if _article_contour_process.poll() is None:
        _article_contour_process.terminate()
        try:
            _article_contour_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _article_contour_process.kill()
            _article_contour_process.wait(timeout=3)
    _article_contour_process = None


def browser_url() -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{http_port}'

print('Equivalent terminal command:')
print(shlex.join(panel_command()))


Equivalent terminal command:
/home/orin/JetCar/.venv/bin/python /home/orin/JetCar/scripts/teleop_server_article_contour_panel.py --host 0.0.0.0 --http-port 8765 --camera-source csi --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port /dev/ttyTHS1 --baudrate 115200


In [11]:
stop_article_contour_panel()
cmd = panel_command()
print('Starting article contour follow panel...')
print(shlex.join(cmd))
with open(panel_log_path, 'w') as log_file:
    _article_contour_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
time.sleep(2.0)
if _article_contour_process.poll() is not None:
    raise RuntimeError(f'article contour panel exited early; check {panel_log_path}')
print('Article contour panel running at:', browser_url())
if ip_addresses:
    print('LAN example:', f'http://{ip_addresses[0]}:{http_port}')
display(HTML(
    f'<p><a href="{browser_url()}" target="_blank">Open the contour follow panel in a new tab</a></p>'
    f'<iframe src="{browser_url()}" width="100%" height="840" style="border:1px solid #ccc; border-radius:12px;"></iframe>'
))


Starting article contour follow panel...
/home/orin/JetCar/.venv/bin/python /home/orin/JetCar/scripts/teleop_server_article_contour_panel.py --host 0.0.0.0 --http-port 8765 --camera-source csi --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port /dev/ttyTHS1 --baudrate 115200
Article contour panel running at: http://127.0.0.1:8765
LAN example: http://127.0.1.1:8765


In [12]:
print('Last contour follow panel log lines:')
if panel_log_path.exists():
    print('\n'.join(panel_log_path.read_text().splitlines()[-40:]))
else:
    print('No log file yet.')


Last contour follow panel log lines:
GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode 

In [13]:
# stop_article_contour_panel()
# print('Article contour panel stopped.')


## What To Do In The Panel

1. make sure `Camera` becomes ready
2. make sure `Serial` becomes ready
3. use the preset test buttons to confirm the saved calibration still feels correct
4. adjust the detection sliders only if the contour or mask looks unstable
5. place the rover on the line and press `Auto Start`
6. watch the `Detection Metrics` and the contour overlay while it drives
7. press `Stop Auto` or `Stop Motors` immediately if the behavior looks wrong

This panel keeps the workflow focused on the article-style contour detector instead of the older teleop mask workflow.
